In [10]:
import numpy
from alternet.annotation import *
from alternet.data_preprocessing import standardize_dataframe
import numpy
import os
import os.path as op






In [2]:


data_path = "/data/bionets/og86asub/alternet-project/alternet/data"
results_path = "/data/bionets/og86asub/alternet-project/alternet/results-2.0"

# Reference files
appris_path = "appris_data.appris.txt"
digger_path = "digger_data.csv"
biomart_path = "biomart.txt"
tf_list_path = "allTFs_hg38.txt"
sf_list_path = "splicefactors.csv"

# Expression data
gtex_transcript_tpm_path = "GTEx_Analysis_v10_RSEMv1.3.3_transcripts_tpm.txt"
gtex_sample_attributes_path = "GTEx_Analysis_v10_Annotations_SampleAttributesDS.txt"

# Tissue to analyze
TISSUE = "Bladder"
CONDITION = TISSUE

# Number of GRNBoost2 runs
N_RUNS = 10

os.makedirs(results_path, exist_ok=True)



In [12]:
from alternet.gtex_dataloader import *

In [17]:
sample_meta = pd.read_csv(op.join(gtex_data_dir, 'GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt'), sep='\t')
sub = sample_meta.loc[:, ['SAMPID', 'SMTS']] # SMTS == tissue type


In [19]:
sub.SMTS.unique()

array(['Blood', 'Brain', 'Adipose Tissue', 'Muscle', 'Blood Vessel',
       'Heart', 'Ovary', 'Uterus', 'Vagina', 'Breast', 'Skin',
       'Salivary Gland', 'Adrenal Gland', 'Thyroid', 'Lung', 'Spleen',
       'Pancreas', 'Esophagus', 'Stomach', 'Colon', 'Small Intestine',
       'Prostate', 'Testis', 'Nerve', 'Pituitary', 'Liver', 'Kidney',
       'Cervix Uteri', 'Fallopian Tube', 'Bladder', 'Bone Marrow'],
      dtype=object)

In [13]:
tissue_ids = retrieve_GTEX_tissue_sampleids(params['sample_attributes'], tissue=params['tissue'])


NameError: name 'params' is not defined

In [8]:
pd.read_csv(op.join(gtex_data_dir, 'GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt'))

ParserError: Error tokenizing data. C error: Expected 1 fields in line 5, saw 2


In [ ]:
biomart = pd.read_csv(op.join(data_path, biomart_path), sep='\t')
tx2gene = dict(zip(biomart['Transcript stable ID'], biomart['Gene stable ID']))
gene2tx = biomart.groupby('Gene stable ID')['Transcript stable ID'].apply(set).to_dict()
appris_df = pd.read_csv(op.join(data_path,appris_path), sep='\t')
digger_df = pd.read_csv(op.join(data_path,digger_path), low_memory=False)
# Load and map TF list
tf_list_raw = pd.read_csv(op.join(data_path,tf_list_path), sep='\t', header=None)
tf_list = map_tf_ids(tf_list_raw, biomart)

In [4]:


VARIANCE_PERCENTILE = 0.7  # Keep top 30%

In [6]:
# Load and map SF list
sf_list_raw = pd.read_csv(op.join(data_path, sf_list_path), header=0, sep = ',')
sf_list = map_sf_ids(sf_list_raw.loc[:, ['Splicing_Factor']], biomart)
# Combine TF and SF lists
regulator_list = combine_tf_sf_lists(tf_list, sf_list)
tx_to_regtype = dict(zip(regulator_list['Transcript stable ID'], regulator_list['Regulator_type']))
gene_to_regtype = regulator_list.groupby('Gene stable ID')['Regulator_type'].first().to_dict()


In [13]:


# Create mappings
transcript_mapper = create_transcript_mapping(biomart)
print(f"Transcript-to-gene mappings: {len(transcript_mapper)}")

# Annotation databases
tf_database = create_transcipt_annotation_database(
    tf_list=tf_list, appris_df=appris_df, digger=digger_df
)
regulator_database = create_transcipt_annotation_database(
    tf_list=regulator_list, appris_df=appris_df, digger=digger_df
)
print(f"TF annotation database: {len(tf_database)} entries")
print(f"Regulator annotation database: {len(regulator_database)} entries")



Transcript-to-gene mappings: 278220
TF annotation database: 16298 entries
Regulator annotation database: 18958 entries


In [ ]:
from alternet.data_preprocessing import *
from alternet.gtex_dataloader import *

In [22]:
gtex_data_dir = '/data/bionets/datasets/hackathon/data/GTEX'
params = {'sample_attributes': op.join(gtex_data_dir, 'GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt'), 'tissue': 'Liver', 'transcript_data':op.join(gtex_data_dir, 'GTEx_Analysis_2017-06-05_v8_RSEMv1.3.0_transcript_tpm.gct')}
tissue_ids = retrieve_GTEX_tissue_sampleids(params['sample_attributes'], tissue=params['tissue'])
transcript_data = read_GTEX_transcript_expression(params['transcript_data'], tissue_ids)
transcript_data = clean_GTEX_tissue_transcript_counts(transcript_data, biomart)
transcript_data = variance_filtering(transcript_data)


Retrieving tissue sample IDs
Reading Transcript expression data
Cleaning up counts


In [31]:

sample_cols = [c for c in transcript_data.columns if c not in ['transcript_id', 'gene_id']]
gene_data = transcript_data.groupby('gene_id')[sample_cols].sum().reset_index()



Gene-level data: 5832 genes


In [34]:

# Create expression matrices (samples × features)
gene_data_matrix = gene_data.set_index('gene_id')[sample_cols].T
transcript_data_matrix = transcript_data.set_index('transcript_id')[sample_cols].T

gene_data_scaled = standardize_dataframe(gene_data_matrix)
transcript_data_scaled = standardize_dataframe(transcript_data_matrix)



Gene matrix: (226, 5832) (samples × genes)
Transcript matrix: (226, 12005) (samples × transcripts)


In [ ]:
transcript_data_scaled, gene_data_scaled = remove_problematic_transcripts(transcript_data_scaled, gene_data_scaled)

In [40]:

# Gene-to-transcript mapping for genes in data
gene_to_transcript_mapping = create_filtered_gene_to_transcripts_mapping(
    biomart,
    gene_list=gene_data_scaled.columns,
    transcript_list=transcript_data_scaled.columns
)

In [42]:


# TF genes in data (for Network 1)
tf_genes_in_data = list(set(tf_list['Gene stable ID']) & set(gene_data_scaled.columns))
print(f"TF genes in data: {len(tf_genes_in_data)}")

# TF transcripts in data (for Network 2)
tf_transcripts_in_data = list(set(tf_list['Transcript stable ID']) & set(transcript_data_scaled.columns))
print(f"TF transcripts in data: {len(tf_transcripts_in_data)}")

# All regulator transcripts (for Network 3)
regulator_transcripts_in_data = list(
    set(regulator_list['Transcript stable ID']) & set(transcript_data_scaled.columns)
)
print(f"All regulator transcripts (TF+SF) in data: {len(regulator_transcripts_in_data)}")

# Targets
target_genes = list(gene_data_scaled.columns)
target_transcripts = list(transcript_data_scaled.columns)

TF genes in data: 391
TF transcripts in data: 824
All regulator transcripts (TF+SF) in data: 1089


In [51]:
from alternet import postprocessing


/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/numba/core/decorators.py:248: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


In [45]:


# TF isoform categories (for Network 1 & 2 comparison)
tf_isoform_categories = postprocessing.isoform_categorization(
    transcript_data_matrix, gene_data_matrix, tf_list
)
tf_gene_categories = postprocessing.get_gene_cases(tf_isoform_categories)



TF isoform categories:
isoform_category
non-dominant    365
balanced        225
single          205
dominant         29
Name: count, dtype: int64


In [52]:
# Regulator isoform categories (for Network 3)
regulator_isoform_categories = postprocessing.isoform_categorization(
    transcript_data_matrix, gene_data_matrix, regulator_list
)
regulator_gene_categories = postprocessing.get_gene_cases(regulator_isoform_categories)

print("Regulator isoform categories:")
print(regulator_isoform_categories['isoform_category'].value_counts())


Regulator isoform categories:
isoform_category
non-dominant    450
balanced        353
single          248
dominant         38
Name: count, dtype: int64


In [47]:


# Target isoform categories
target_list = biomart[
    biomart['Transcript stable ID'].isin(transcript_data_scaled.columns)
][['Gene stable ID', 'Transcript stable ID']].drop_duplicates()

target_isoform_categories = postprocessing.isoform_categorization(
    transcript_data_matrix, gene_data_matrix, target_list
)
target_gene_categories = postprocessing.get_gene_cases(target_isoform_categories)

print("Target isoform categories:")
print(target_isoform_categories['isoform_category'].value_counts())



Target isoform categories:
isoform_category
non-dominant    5142
balanced        3304
single          3006
dominant         523
Name: count, dtype: int64


In [ ]:

import time
from alternet.inference import inference

runtime = {}

start = time.monotonic()
canonical_grn = inference(
    gene_data=gene_data_scaled,
    tf_list=tf_genes_in_data,
    target_names='all',
    n_runs=N_RUNS
)
runtime['canonical'] = time.monotonic() - start

canonical_grn = canonical_grn.rename(columns={'source': 'source_gene', 'target': 'target_gene'})
canonical_grn['reg_type'] = canonical_grn['source_gene'].map(gene_to_regtype)
canonical_grn.to_csv(op.join(results_path, f"{CONDITION}_canonical_raw.tsv"), sep='\t', index=False)



NETWORK 1: Canonical
Regulators: 391 TF genes
Targets: 5832 genes


  0%|          | 0/10 [00:00<?, ?it/s]/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 12.50 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
 10%|█         | 1/10 [02:44<24:41, 164.65s/it]/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 12.50 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings


Edges: 2,276,454
Time: 27.38 minutes


In [ ]:

# Create hybrid data (TF transcripts + target genes)
hybrid_data = create_hybrid_data(
    transcript_data_matrix,  
    gene_data_matrix,        
    tf_list
)

start = time.monotonic()
as_source_grn = inference(
    gene_data=hybrid_data,
    tf_list=tf_transcripts_in_data,
    target_names=target_genes,
    n_runs=N_RUNS
)
runtime['as_aware_source'] = time.monotonic() - start

as_source_grn = as_source_grn.rename(columns={'source': 'source_transcript', 'target': 'target_gene'})
as_source_grn['source_gene'] = as_source_grn['source_transcript'].map(tx2gene)
as_source_grn['reg_type'] = as_source_grn['source_transcript'].map(tx_to_regtype)

as_source_grn.to_csv(op.join(results_path, f"{CONDITION}_as_aware_source_raw.tsv"), sep='\t', index=False)



NETWORK 2: AS-Aware Source
Regulators: 824 TF transcripts
Targets: 5832 genes
Hybrid data shape: (226, 6656)


/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 46059 instead
  warnings.warn(
  0%|          | 0/10 [00:00<?, ?it/s]/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 12.50 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
 10%|█         | 1/10 [03:16<29:32, 196.94s/it]/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 12.50 MiB.
This may


Edges: 4,772,829
Time: 32.93 minutes


In [ ]:

start = time.monotonic()
fully_as_grn = inference(
    gene_data=transcript_data_scaled,
    tf_list=regulator_transcripts_in_data,
    target_names='all',
    n_runs=N_RUNS
)
runtime['fully_as_aware'] = time.monotonic() - start


# rename columns
fully_as_grn = fully_as_grn.rename(columns={'source': 'source_transcript', 'target': 'target_transcript'})
fully_as_grn['source_gene'] = fully_as_grn['source_transcript'].map(tx2gene)
fully_as_grn['target_gene'] = fully_as_grn['target_transcript'].map(tx2gene)
fully_as_grn['reg_type'] = fully_as_grn['source_transcript'].map(tx_to_regtype)
fully_as_grn.to_csv(op.join(results_path, f"{CONDITION}_fully_as_aware_raw.tsv"), sep='\t', index=False)



NETWORK 3: Fully AS-Aware
Regulators: 1089 transcripts (TF+SF)
Targets: 12005 transcripts


/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 42085 instead
  warnings.warn(
  0%|          | 0/10 [00:00<?, ?it/s]/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 25.74 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(
 10%|█         | 1/10 [07:02<1:03:21, 422.44s/it]/data/bionets/og86asub/alternet-project/alternet/.pixi/envs/kernel/lib/python3.11/site-packages/distributed/client.py:3370: UserWarning: Sending large graph of size 25.74 MiB.
This m


Edges: 12,899,677
Time: 71.05 minutes


In [ ]:
import yaml
def write_dict_to_yaml(data, filepath):
    """Write dictionary to YAML file."""
    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)

runtime['total'] = runtime['canonical'] + runtime['as_aware_source'] + runtime['fully_as_aware']
write_dict_to_yaml(runtime, op.join(results_path, f"{CONDITION}_runtime.yaml"))import yaml
def write_dict_to_yaml(data, filepath):
    """Write dictionary to YAML file."""
    with open(filepath, 'w') as f:
        yaml.dump(data, f, default_flow_style=False)

runtime['total'] = runtime['canonical'] + runtime['as_aware_source'] + runtime['fully_as_aware']
write_dict_to_yaml(runtime, op.join(results_path, f"{CONDITION}_runtime.yaml"))
print(f"\nTotal inference time: {runtime['total']/60:.2f} minutes")

print(f"\nTotal inference time: {runtime['total']/60:.2f} minutes")



Total inference time: 131.36 minutes


In [ ]:
# Filtering Parameters
MIN_FREQUENCY = 10
IMPORTANCE_PERCENTILE = 0.7  # Keep top 30%


# Set C and Set D: PSI/Usage thresholds

DOM_MIN = 0.5
DOM_EQ_MIN = 0.7
GENE_TPM_MIN = 1.0,

# Set C specific
MIN_ISOFORMS_FOR_SPLICING = 2
TOP_M_EXPRESSED = 3



In [1]:
from alternet.postprocessing import *

In [68]:
canonical_grn_filtered = filter_edges(canonical_grn, MIN_FREQUENCY, IMPORTANCE_PERCENTILE)
as_source_grn_filtered = filter_edges(as_source_grn, MIN_FREQUENCY, IMPORTANCE_PERCENTILE)
fully_as_grn_filtered = filter_edges(fully_as_grn, MIN_FREQUENCY, IMPORTANCE_PERCENTILE)


Network 1 (Canonical)

Network 2 (AS-Aware Source)

Network 3 (Fully AS-Aware)
  Before: 12,899,677
  After frequency filter (>=10): 2,067,511
  After importance filter (top 30%, threshold=0.7697): 620,254


In [90]:
net1_tf = canonical_grn_filtered[canonical_grn_filtered['reg_type'].isin(['TF', 'TF_SF'])].copy()
net2_tf = as_source_grn_filtered[as_source_grn_filtered['reg_type'].isin(['TF', 'TF_SF'])].copy()


In [97]:
set_a = canonical_vs_source_as(net1_tf,net2_tf)


Set A: 454,849 rows

Category distribution:
  source_gene_specific            257,769 ( 56.7%)
  source_isoform_specific         102,998 ( 22.6%)
  source_ambiguous                 50,358 ( 11.1%)
  source_equivalent                43,724 (  9.6%)


In [159]:
usage_df, reliability_df, metadata = calculate_transcript_usage(transcript_data.reset_index())

In [ ]:
usage_df = usage_df.set_index('transcript_id')[sample_cols]
transcript_data = transcript_data.set_index('transcript_id')[sample_cols]

In [100]:
# Split Network 3 by regulator type
net3_tf = fully_as_grn_filtered[fully_as_grn_filtered['reg_type'] == 'TF'].copy()
net3_sf = fully_as_grn_filtered[fully_as_grn_filtered['reg_type'] == 'SF'].copy()
net3_tfsf = fully_as_grn_filtered[fully_as_grn_filtered['reg_type'] == 'TF_SF'].copy()


In [ ]:
# Split Network 3 by regulator type
net3_tf = fully_as_grn_filtered[fully_as_grn_filtered['reg_type'] == 'TF'].copy()
net3_sf = fully_as_grn_filtered[fully_as_grn_filtered['reg_type'] == 'SF'].copy()
net3_tfsf = fully_as_grn_filtered[fully_as_grn_filtered['reg_type'] == 'TF_SF'].copy()


In [174]:
set_d_full = tf_sf_disambigouation_fully_as_aware(net3_tfsf, regulator_list, transcript_data, usage_df, reliability_df)

In [175]:
print("\nSet D - Initial TF+SF Categories:")
for cat, count in set_d_full['tfsf_category'].value_counts().items():
    print(f"  {cat:25} {count:>8,} ({count/len(set_d_full)*100:>5.1f}%)")

# Split for downstream processing
tfsf_tf_like = set_d_full[set_d_full['tfsf_category'] == 'tfsf_tf_like'].copy()
tfsf_sf_like = set_d_full[set_d_full['tfsf_category'] == 'tfsf_sf_like'].copy()
tfsf_joint = set_d_full[set_d_full['tfsf_category'] == 'tfsf_joint'].copy()
tfsf_ambiguous = set_d_full[set_d_full['tfsf_category'] == 'tfsf_ambiguous'].copy()

print(f"\n  tfsf_tf_like: {len(tfsf_tf_like):,} (will be added to Set B)")
print(f"  tfsf_sf_like: {len(tfsf_sf_like):,} (will be added to Set C)")
print(f"  tfsf_joint: {len(tfsf_joint):,} (stays in Set D)")
print(f"  tfsf_ambiguous: {len(tfsf_ambiguous):,} (stays in Set D)")


Set D - Initial TF+SF Categories:
  tfsf_ambiguous                  46 ( 62.2%)
  tfsf_sf_like                    23 ( 31.1%)
  tfsf_tf_like                     5 (  6.8%)

  tfsf_tf_like: 5 (will be added to Set B)
  tfsf_sf_like: 23 (will be added to Set C)
  tfsf_joint: 0 (stays in Set D)
  tfsf_ambiguous: 46 (stays in Set D)


In [189]:
def categorize_target_resolution(edge_tg, net2_mean, net3_mean, net3_max, net3_dom,
                                  net2_median, net3_median,
                                  r_iso=R_ISO, r_gene=R_GENE, r_eq=R_EQ, eps=EPSILON):
    """
    Categorize edge based on target resolution.
    Uses MEAN importance for ratio calculation (AlterNet 1.0 style).
    """
    S2_mean = net2_mean.get(edge_tg, 0)
    S3_mean_sum = net3_mean.get(edge_tg, 0)
    S3_mean_max = net3_max.get(edge_tg, 0)
    S2_median = net2_median.get(edge_tg, 0)
    S3_median = net3_median.get(edge_tg, 0)
    dominance = net3_dom.get(edge_tg, 0)
    
    E2, E3 = S2_mean > 0, S3_mean_sum > 0
    reg_tx, target_gene = edge_tg.split('|')
    reg_gene = tx2gene.get(reg_tx, None)
    
    # Use MEAN importance for fold-change calculation
    if E3 and not E2:
        category = 'target_isoform_specific'
    elif E2 and not E3:
        category = 'target_gene_specific'
    elif E2 and E3:
        ratio = S3_mean_sum / (S2_mean + eps)
        if ratio >= r_iso:
            category = 'target_isoform_specific'
        elif ratio <= 1/r_gene:
            category = 'target_gene_specific'
        elif 1/r_eq <= ratio <= r_eq:
            category = 'target_equivalent'
        else:
            category = 'target_ambiguous'
    else:
        category = 'target_ambiguous'
    
    return {
        'regulator_tx': reg_tx, 'regulator_gene': reg_gene, 'target_gene': target_gene,
        'S2_mean': S2_mean, 'S3_mean_sum': S3_mean_sum, 'S3_mean_max': S3_mean_max,
        'S2_median': S2_median, 'S3_median': S3_median,
        'dominance': dominance,
        'E2': E2, 'E3': E3,
        'ratio': S3_mean_sum / (S2_mean + eps) if E2 else np.inf,
        'target_category': category
    }

In [186]:
# Net2: ALL TF-like edges (TF + TF_SF) - they all act as TFs in gene-level target network
net2_for_t2 = as_source_grn_filtered[as_source_grn_filtered['reg_type'].isin(['TF', 'TF_SF'])].copy()

# Net3: TF only (TF_SF is handled via Set D)
net3_tf_only = fully_as_grn_filtered[fully_as_grn_filtered['reg_type'] == 'TF'].copy()

print(f"Net2 (TF + TF_SF acting as TF): {len(net2_for_t2):,}")
print(f"Net3 (TF only): {len(net3_tf_only):,}")



Net2 (TF + TF_SF acting as TF): 361,708
Net3 (TF only): 444,700


In [187]:


# Prepare Net2: transcript-gene level
net2_for_t2['edge_tg'] = net2_for_t2['source_transcript'] + '|' + net2_for_t2['target_gene']
net2_tg_mean_imp = dict(zip(net2_for_t2['edge_tg'], net2_for_t2['mean_importance']))
net2_tg_median_imp = dict(zip(net2_for_t2['edge_tg'], net2_for_t2['median_importance']))

# Prepare Net3 TF-only: aggregate to transcript-gene level
net3_tf_only['edge_tg'] = net3_tf_only['source_transcript'] + '|' + net3_tf_only['target_gene']
net3_tg_agg = net3_tf_only.groupby('edge_tg').agg({
    'mean_importance': ['sum', 'max', 'count'],
    'median_importance': ['sum', 'max'],
    'source_gene': 'first',
    'target_gene': 'first',
    'target_transcript': lambda x: ','.join(sorted(set(x)))
}).reset_index()
net3_tg_agg.columns = ['edge_tg', 'S3_mean_sum', 'S3_mean_max', 'n_target_tx', 
                       'S3_median_sum', 'S3_median_max', 'source_gene', 'target_gene',
                       'target_tx_set']
net3_tg_agg['dominance'] = net3_tg_agg['S3_mean_max'] / (net3_tg_agg['S3_mean_sum'] + EPSILON)

net3_tg_mean_imp = dict(zip(net3_tg_agg['edge_tg'], net3_tg_agg['S3_mean_sum']))
net3_tg_mean_max = dict(zip(net3_tg_agg['edge_tg'], net3_tg_agg['S3_mean_max']))
net3_tg_median_imp = dict(zip(net3_tg_agg['edge_tg'], net3_tg_agg['S3_median_sum']))
net3_tg_dom = dict(zip(net3_tg_agg['edge_tg'], net3_tg_agg['dominance']))

# All edges for TF part of Set B
all_edges_t2_tf = set(net2_tg_mean_imp.keys()) | set(net3_tg_mean_imp.keys())
print(f"\nTF edges (Net2 TF+TF_SF vs Net3 TF): {len(all_edges_t2_tf):,}")




TF edges (Net2 TF+TF_SF vs Net3 TF): 492,354


In [190]:
# Categorize TF edges (from Net2 TF+TF_SF vs Net3 TF)
set_b_tf_rows = [categorize_target_resolution(e, net2_tg_mean_imp, net3_tg_mean_imp, 
                                                net3_tg_mean_max, net3_tg_dom,
                                                net2_tg_median_imp, net3_tg_median_imp) 
                  for e in all_edges_t2_tf]
set_b_tf = pd.DataFrame(set_b_tf_rows)
set_b_tf['reg_type'] = 'TF'

# Attach target_tx_set from Net3 aggregation
_net3_tx_map = dict(zip(net3_tg_agg['edge_tg'], net3_tg_agg['target_tx_set']))
set_b_tf['target_tx_set'] = set_b_tf.apply(
    lambda r: _net3_tx_map.get(r['regulator_tx'] + '|' + r['target_gene'], ''), axis=1)

print(f"Set B base (TF): {len(set_b_tf):,} rows")

Set B base (TF): 492,354 rows


In [192]:
# Now add tfsf_tf_like edges
if len(tfsf_tf_like) > 0:
    
    # Get Net2 TF_SF edges (for comparison)
    net2_tfsf = as_source_grn_filtered[as_source_grn_filtered['reg_type'] == 'TF_SF'].copy()
    net2_tfsf['edge_tg'] = net2_tfsf['source_transcript'] + '|' + net2_tfsf['target_gene']
    net2_tfsf_mean_imp = dict(zip(net2_tfsf['edge_tg'], net2_tfsf['mean_importance']))
    net2_tfsf_median_imp = dict(zip(net2_tfsf['edge_tg'], net2_tfsf['median_importance']))
    
    # Get Net3 TF_SF edges aggregated (for the tf_like subset)
    net3_tfsf_agg = net3_tfsf.groupby(['source_transcript', 'target_gene']).agg({
        'mean_importance': ['sum', 'max', 'count'],
        'median_importance': ['sum', 'max'],
        'source_gene': 'first',
        'target_transcript': lambda x: ','.join(sorted(set(x)))
    }).reset_index()
    net3_tfsf_agg.columns = ['source_transcript', 'target_gene', 'S3_mean_sum', 'S3_mean_max', 'n_target_tx',
                            'S3_median_sum', 'S3_median_max', 'source_gene', 'target_tx_set']
    net3_tfsf_agg['edge_tg'] = net3_tfsf_agg['source_transcript'] + '|' + net3_tfsf_agg['target_gene']
    net3_tfsf_agg['dominance'] = net3_tfsf_agg['S3_mean_max'] / (net3_tfsf_agg['S3_mean_sum'] + EPSILON)
    
    tfsf_tg_mean_imp = dict(zip(net3_tfsf_agg['edge_tg'], net3_tfsf_agg['S3_mean_sum']))
    tfsf_tg_mean_max = dict(zip(net3_tfsf_agg['edge_tg'], net3_tfsf_agg['S3_mean_max']))
    tfsf_tg_median_imp = dict(zip(net3_tfsf_agg['edge_tg'], net3_tfsf_agg['S3_median_sum']))
    tfsf_tg_dom = dict(zip(net3_tfsf_agg['edge_tg'], net3_tfsf_agg['dominance']))
    
    # Get unique edges from tfsf_tf_like
    tfsf_tf_like['edge_tg'] = tfsf_tf_like['reg_tx'] + '|' + tfsf_tf_like['target_gene']
    tfsf_tf_like_edges = set(tfsf_tf_like['edge_tg'].unique())
    
    # Categorize tfsf_tf_like edges
    set_b_tfsf_rows = [categorize_target_resolution(e, net2_tfsf_mean_imp, tfsf_tg_mean_imp, 
                                                      tfsf_tg_mean_max, tfsf_tg_dom,
                                                      net2_tfsf_median_imp, tfsf_tg_median_imp) 
                        for e in tfsf_tf_like_edges]
    set_b_tfsf = pd.DataFrame(set_b_tfsf_rows)
    set_b_tfsf['reg_type'] = 'TF_SF'
    
    # Attach target_tx_set
    _tfsf_tx_map = dict(zip(
        net3_tfsf_agg['source_transcript'] + '|' + net3_tfsf_agg['target_gene'],
        net3_tfsf_agg['target_tx_set']))
    set_b_tfsf['target_tx_set'] = set_b_tfsf.apply(
        lambda r: _tfsf_tx_map.get(r['regulator_tx'] + '|' + r['target_gene'], ''), axis=1)
    
else:
    set_b_tfsf = pd.DataFrame()

In [193]:
# Combine into final Set B
set_b = pd.concat([set_b_tf, set_b_tfsf], ignore_index=True)

# Sort by median importance (AlterNet 1.0 style)
set_b['max_median'] = set_b[['S2_median', 'S3_median']].max(axis=1)
set_b = set_b.sort_values('max_median', ascending=False)

print(f"\nFinal Set B: {len(set_b):,} rows")
print(f"  - TF edges (from Net2 TF+TF_SF vs Net3 TF): {len(set_b_tf):,}")
print(f"  - TF_SF (tf_like from Set D): {len(set_b_tfsf) if len(set_b_tfsf) > 0 else 0:,}")
print("\nCategory distribution:")
for cat, count in set_b['target_category'].value_counts().items():
    print(f"  {cat:30} {count:>8,} ({count/len(set_b)*100:>5.1f}%)")
print("\nBy regulator type:")
print(set_b.groupby(['reg_type', 'target_category']).size().unstack(fill_value=0))


Final Set B: 492,359 rows
  - TF edges (from Net2 TF+TF_SF vs Net3 TF): 492,354
  - TF_SF (tf_like from Set D): 5

Category distribution:
  target_isoform_specific         183,186 ( 37.2%)
  target_gene_specific            182,820 ( 37.1%)
  target_equivalent                68,743 ( 14.0%)
  target_ambiguous                 57,610 ( 11.7%)

By regulator type:
target_category  target_ambiguous  target_equivalent  target_gene_specific  \
reg_type                                                                     
TF                          57608              68743                182820   
TF_SF                           2                  0                     0   

target_category  target_isoform_specific  
reg_type                                  
TF                                183183  
TF_SF                                  3  


In [202]:

net3_tf_unpack = net3_tf_only[['source_transcript', 'source_gene', 'target_transcript',
                                'target_gene', 'mean_importance', 'median_importance', 
                                'frequency']].copy()
net3_tf_unpack.columns = ['regulator_tx', 'regulator_gene', 'target_tx', 'target_gene',
                           'net3_mean_importance', 'net3_median_importance', 'net3_frequency']
net3_tf_unpack['reg_type'] = 'TF'

# Source 2: tfsf_tf_like edges from net3_tfsf
if len(tfsf_tf_like) > 0:
    tfsf_tf_like_regtx_set = set(tfsf_tf_like['reg_tx'].unique())
    net3_tfsf_unpack = net3_tfsf[
        net3_tfsf['source_transcript'].isin(tfsf_tf_like_regtx_set)
    ][['source_transcript', 'source_gene', 'target_transcript',
       'target_gene', 'mean_importance', 'median_importance', 
       'frequency']].copy()
    net3_tfsf_unpack.columns = ['regulator_tx', 'regulator_gene', 'target_tx', 'target_gene',
                                 'net3_mean_importance', 'net3_median_importance', 'net3_frequency']
    net3_tfsf_unpack['reg_type'] = 'TF_SF'
    set_b_unpacked = pd.concat([net3_tf_unpack, net3_tfsf_unpack], ignore_index=True)
else:
    set_b_unpacked = net3_tf_unpack.copy()

# Join with Set B categorization
# Create join key
set_b_unpacked['edge_tg'] = set_b_unpacked['regulator_tx'] + '|' + set_b_unpacked['target_gene']
set_b_key = set_b[['regulator_tx', 'target_gene', 'target_category', 'dominance',
                      'S3_mean_sum', 'target_tx_set']].copy()
set_b_key['edge_tg'] = set_b_key['regulator_tx'] + '|' + set_b_key['target_gene']

# Merge: only keep edges whose parent exists in Set B
set_b_unpacked = set_b_unpacked.merge(
    set_b_key[['edge_tg', 'target_category']],
    on='edge_tg', how='inner'
)

# Add target_rank_within_edge
set_b_unpacked['target_rank_within_edge'] = set_b_unpacked.groupby('edge_tg')[
    'net3_mean_importance'].rank(ascending=False, method='min').astype(int)

# Sort by importance
set_b_unpacked = set_b_unpacked.sort_values('net3_median_importance', ascending=False)

# Drop join key
set_b_unpacked = set_b_unpacked.drop(columns=['edge_tg'])

print(f"Set B Unpacked: {len(set_b_unpacked):,} transcript-level edges")
print(f"  From TF edges: {(set_b_unpacked['reg_type'] == 'TF').sum():,}")
print(f"  From TF_SF (tf_like): {(set_b_unpacked['reg_type'] == 'TF_SF').sum():,}")
print(f"\nCategory distribution (inherited):")
for cat, count in set_b_unpacked['target_category'].value_counts().items():
    print(f"  {cat:30} {count:>8,} ({count/len(set_b_unpacked)*100:>5.1f}%)")
print(f"\nRank distribution:")
print(f"  Rank 1 (dominant target tx): {(set_b_unpacked['target_rank_within_edge'] == 1).sum():,}")
print(f"  Rank 2+: {(set_b_unpacked['target_rank_within_edge'] > 1).sum():,}")

Set B Unpacked: 447,116 transcript-level edges
  From TF edges: 444,700
  From TF_SF (tf_like): 2,416

Category distribution (inherited):
  target_isoform_specific         254,521 ( 56.9%)
  target_equivalent                76,353 ( 17.1%)
  target_ambiguous                 67,269 ( 15.0%)
  target_gene_specific             48,973 ( 11.0%)

Rank distribution:
  Rank 1 (dominant target tx): 355,351
  Rank 2+: 91,765


In [208]:
as_source_grn_filtered

,source_transcript,target_gene,frequency,mean_importance,median_importance,source_gene,reg_type
5,ENST00000020945,ENSG00000001630,10,1.899626,1.859785,ENSG00000019549,TF
7,ENST00000020945,ENSG00000002330,10,0.717990,0.672066,ENSG00000019549,TF
22,ENST00000020945,ENSG00000004468,10,6.200406,6.337433,ENSG00000019549,TF
27,ENST00000020945,ENSG00000004776,10,1.032156,1.044973,ENSG00000019549,TF
29,ENST00000020945,ENSG00000004799,10,3.957560,4.136152,ENSG00000019549,TF
...,...,...,...,...,...,...,...
4772803,ENST00000640075,ENSG00000275052,10,3.217007,3.088827,ENSG00000169764,TF
4772812,ENST00000640075,ENSG00000276490,10,1.373983,1.177988,ENSG00000169764,TF
4772813,ENST00000640075,ENSG00000276911,10,3.839126,4.118168,ENSG00000169764,TF
4772820,ENST00000640075,ENSG00000277893,10,0.784800,0.902015,ENSG00000169764,TF


In [ ]:
as_source_grn_filtered[(as_source_grn_filtered.source_transcript == 'ENST00000354725') & (as_source_grn_filtered.target_gene =='ENSG00000000938')]

,source_transcript,target_gene,frequency,mean_importance,median_importance,source_gene,reg_type
986465,ENST00000354725,ENSG00000123131,10,39.956081,36.139492,ENSG00000197157,TF_SF


In [224]:
as_source_grn_filtered[(as_source_grn_filtered.source_gene == 'ENSG00000124216') & (as_source_grn_filtered.target_gene =='ENSG00000000938')]

,source_transcript,target_gene,frequency,mean_importance,median_importance,source_gene,reg_type
144792,ENST00000244050,ENSG00000000938,10,12.499045,11.834214,ENSG00000124216,TF


In [221]:
set_b_unpacked[(set_b_unpacked.target_category == 'target_gene_specific') & (set_b_unpacked.regulator_gene!=set_b_unpacked.target_gene)].sort_values('target_gene')

,regulator_tx,regulator_gene,target_tx,target_gene,net3_mean_importance,net3_median_importance,net3_frequency,reg_type,target_category,target_rank_within_edge
379899,ENST00000547773,ENSG00000187109,ENST00000374004,ENSG00000000938,1.001264,0.933628,10,TF,target_gene_specific,1
76128,ENST00000324460,ENSG00000044574,ENST00000374004,ENSG00000000938,4.322616,4.638274,10,TF,target_gene_specific,1
19033,ENST00000244050,ENSG00000124216,ENST00000374005,ENSG00000000938,1.326108,1.042943,10,TF,target_gene_specific,2
2073,ENST00000189444,ENSG00000077150,ENST00000374005,ENSG00000000938,0.755206,0.774210,10,TF,target_gene_specific,1
19032,ENST00000244050,ENSG00000124216,ENST00000374004,ENSG00000000938,5.139328,5.339022,10,TF,target_gene_specific,1
...,...,...,...,...,...,...,...,...,...,...
303969,ENST00000483882,ENSG00000025434,ENST00000636204,ENSG00000283189,1.244786,1.150602,10,TF,target_gene_specific,1
375958,ENST00000541679,ENSG00000132341,ENST00000636204,ENSG00000283189,0.980213,0.964187,10,TF,target_gene_specific,1
268349,ENST00000467728,ENSG00000025434,ENST00000636204,ENSG00000283189,1.492945,1.459576,10,TF,target_gene_specific,1
239115,ENST00000440176,ENSG00000060971,ENST00000636204,ENSG00000283189,1.436419,1.380499,10,TF,target_gene_specific,1


In [227]:
fully_as_grn[(fully_as_grn.source_gene == 'ENSG00000124216') & (fully_as_grn.target_gene =='ENSG00000000938')]

,source_transcript,target_transcript,frequency,mean_importance,median_importance,source_gene,target_gene
358445,ENST00000244050,ENST00000374004,10,5.139328,5.339022,ENSG00000124216,ENSG00000000938
358446,ENST00000244050,ENST00000374005,10,1.326108,1.042943,ENSG00000124216,ENSG00000000938
